In [7]:
# =========================================================
# DEEPFAKE IMAGE DETECTION PROJECT
# FULL SINGLE FILE CODE FOR VS CODE
# =========================================================

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator
)

from tensorflow.keras.preprocessing import image

from tensorflow.keras.applications import MobileNetV2

from tensorflow.keras import layers, models


# =========================================================
# DATASET PATH
# =========================================================

# Dataset folder structure:
#
# dataset/
#     fake/
#     real/
#     other/

dataset_path = r"C:\Users\Surya\evolve\deep2\dataset"


# =========================================================
# CHECK DATASET
# =========================================================

print("\nChecking Dataset...\n")

if os.path.exists(dataset_path):

    print("Dataset Found")

    print("\nDataset Folders:\n")

    print(os.listdir(dataset_path))

else:

    print("Dataset Path Not Found")

    exit()


# =========================================================
# IMAGE PREPROCESSING
# =========================================================

datagen = ImageDataGenerator(

    rescale=1./255,

    validation_split=0.2
)


# =========================================================
# TRAINING DATA
# =========================================================

train_data = datagen.flow_from_directory(

    dataset_path,

    target_size=(224,224),

    batch_size=32,

    class_mode='categorical',

    subset='training'
)


# =========================================================
# VALIDATION DATA
# =========================================================

val_data = datagen.flow_from_directory(

    dataset_path,

    target_size=(224,224),

    batch_size=32,

    class_mode='categorical',

    subset='validation'
)


# =========================================================
# LOAD MOBILENETV2
# =========================================================

base_model = MobileNetV2(

    input_shape=(224,224,3),

    include_top=False,

    weights='imagenet'
)


# Freeze pretrained layers

base_model.trainable = False


# =========================================================
# BUILD MODEL
# =========================================================

model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(

        128,

        activation='relu'
    ),
    layers.Dense(
    1,
    activation='sigmoid'


    
    )
])


# =========================================================
# COMPILE MODEL
# =========================================================

model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']
)


# =========================================================
# MODEL SUMMARY
# =========================================================

print("\nMODEL SUMMARY\n")

model.summary()


# =========================================================
# TRAIN MODEL
# =========================================================

print("\nTraining Started...\n")

history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=5
)


# =========================================================
# SAVE MODEL
# =========================================================

model.save("model.keras")

print("\nModel Saved Successfully\n")


# =========================================================
# CHECK SAVED MODEL
# =========================================================

if os.path.exists("model.keras"):

    print("Saved Model Found")

else:

    print("Model Save Failed")


# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_image(img_path):

    if not os.path.exists(img_path):

        print("\nImage Not Found\n")

        return

    # Load image

    img = image.load_img(

        img_path,

        target_size=(224,224)
    )

    # Convert image to array

    img_array = image.img_to_array(img)

    # Normalize image

    img_array = img_array / 255.0

    # Expand dimensions

    img_array = np.expand_dims(

        img_array,

        axis=0
    )

    # Prediction

    prediction = model.predict(img_array)

    predicted_class = np.argmax(prediction)

    confidence = np.max(prediction)

    print("\nPrediction Scores:\n")

    print(prediction)

    print("\nConfidence:", confidence)

    print("\nPredicted Class:", predicted_class)

    labels = {

        0: "Fake Image",

        1: "Real Image",

        2: "Other Class"
    }

    result = labels.get(

        predicted_class,

        "Unknown"
    )

    print("\nFINAL RESULT:\n")

    print(result)


# =========================================================
# TEST IMAGE PATH
# =========================================================

test_image_path = r"C:\Users\Surya\evolve\deepfake dector\test.jpg"


# =========================================================
# RUN PREDICTION
# =========================================================

predict_image(test_image_path)


# =========================================================
# PROJECT COMPLETED
# =========================================================

print("\nPROJECT COMPLETED SUCCESSFULLY\n")


Checking Dataset...

Dataset Found

Dataset Folders:

['License-Plate-Data']
Found 347 images belonging to 1 classes.
Found 86 images belonging to 1 classes.

MODEL SUMMARY



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


Training Started...

Epoch 1/5


c:\Users\Surya\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\losses\losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


11/11 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - accuracy: 0.2392 - loss: 0.0000e+00 - val_accuracy: 0.2442 - val_loss: 0.0000e+00
Epoch 2/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.2392 - loss: 0.0000e+00 - val_accuracy: 0.2442 - val_loss: 0.0000e+00
Epoch 3/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.2392 - loss: 0.0000e+00 - val_accuracy: 0.2442 - val_loss: 0.0000e+00
Epoch 4/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.2392 - loss: 0.0000e+00 - val_accuracy: 0.2442 - val_loss: 0.0000e+00
Epoch 5/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.2392 - loss: 0.0000e+00 - val_accuracy: 0.2442 - val_loss: 0.0000e+00

Model Saved Successfully

Saved Model Found

Image Not Found


PROJECT COMPLETED SUCCESSFULLY

